In [37]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score,recall_score,accuracy_score,f1_score,confusion_matrix,classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
df=pd.read_csv('loan_data.csv')
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  

In [4]:
df.isnull().sum()

person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64

In [5]:
df.duplicated().sum()

0

In [6]:
x=df.drop('loan_status',axis=1)
y=df['loan_status']

In [7]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [8]:
xtrain.columns

Index(['person_age', 'person_gender', 'person_education', 'person_income',
       'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'credit_score', 'previous_loan_defaults_on_file'],
      dtype='object')

In [9]:
num_cat=x.select_dtypes('number').columns
cat_col=x.select_dtypes('object').columns

# Method 1

In [10]:
# num_cat=['person_age','person_income','person_emp_exp','loan_amnt','loan_int_rate','loan_percent_income','cb_person_cred_hist_length','credit_score']
# cat_col=['person_gender','person_education','person_home_ownership','loan_intent','previous_loan_defaults_on_file']

In [11]:
preprocessing=ColumnTransformer(transformers=[
    ('encoding',OneHotEncoder(dtype='int',sparse_output=False,handle_unknown='ignore'),cat_col),
    ('scaling',StandardScaler(),num_cat)
])
preprocessing

ColumnTransformer(transformers=[('encoding',
                                 OneHotEncoder(dtype='int',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 Index(['person_gender', 'person_education', 'person_home_ownership',
       'loan_intent', 'previous_loan_defaults_on_file'],
      dtype='object')),
                                ('scaling', StandardScaler(),
                                 Index(['person_age', 'person_income', 'person_emp_exp', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'credit_score'],
      dtype='object'))])

In [12]:
preprocessing.fit_transform(xtrain,ytrain)

array([[ 1.        ,  0.        ,  0.        , ...,  0.11905358,
         0.81673169, -0.03244403],
       [ 0.        ,  1.        ,  0.        , ...,  0.23392088,
        -0.47891374,  0.52468678],
       [ 1.        ,  0.        ,  0.        , ..., -0.79988484,
         1.33498986,  1.00222747],
       ...,
       [ 1.        ,  0.        ,  1.        , ...,  0.46365548,
         0.81673169,  0.38540408],
       [ 0.        ,  1.        ,  0.        , ..., -0.34041563,
        -0.73804283, -0.5696773 ],
       [ 1.        ,  0.        ,  0.        , ..., -0.79988484,
        -0.73804283,  0.62417442]])

In [13]:
pipline=Pipeline(steps=[
    ('Prepro_obj',preprocessing),
    ('model',LogisticRegression(max_iter=1000,solver='lbfgs'))
])

In [14]:
pipline.fit(xtrain,ytrain)

Pipeline(steps=[('Prepro_obj',
                 ColumnTransformer(transformers=[('encoding',
                                                  OneHotEncoder(dtype='int',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  Index(['person_gender', 'person_education', 'person_home_ownership',
       'loan_intent', 'previous_loan_defaults_on_file'],
      dtype='object')),
                                                 ('scaling', StandardScaler(),
                                                  Index(['person_age', 'person_income', 'person_emp_exp', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'credit_score'],
      dtype='object'))])),
                ('model', LogisticRegression(max_iter=1000))])

# Method 2

In [15]:
# encoder=OneHotEncoder(dtype='int',sparse_output=False,handle_unknown='ignore')

In [16]:
# xtrain[encoder.get_feature_names_out()] = encoder.fit_transform(xtrain[cat_col])
# xtest[encoder.get_feature_names_out()] = encoder.transform(xtest[cat_col])

In [17]:
# xtrain.drop(columns=cat_col,inplace=True,axis=1)
# xtest.drop(columns=cat_col,inplace=True,axis=1)

# Method 3

In [18]:
# encode=OneHotEncoder(dtype='int',sparse_output=False,handle_unknown='ignore')

In [19]:
# xtrain(encode.get_feature_names_out(cat_col))=encode.fit_transform(xtrain[cat_col])

In [20]:
# xtrain_encode=encode.fit_transform(xtrain[cat_col])
# encoded_xtrain_df=pd.DataFrame(xtrain_encode,columns=encode.get_feature_names_out(cat_col),index=xtrain.index)
# xtrain=pd.concat([xtrain,encoded_xtrain_df],axis=1)
# xtrain.drop(columns=cat_col,inplace=True)

In [21]:
# xtest_encode=encode.transform(xtest[cat_col])
# encoded_xtest_df=pd.DataFrame(xtest_encode,columns=encode.get_feature_names_out(cat_col),index=xtest.index)
# xtest=pd.concat([xtest,encoded_xtest_df],axis=1)
# xtest.drop(columns=cat_col,inplace=True)

In [22]:
# scale=StandardScaler()

In [23]:
# scaled_xtrain=scale.fit_transform(xtrain[num_cat])
# scaled_xtest=scale.fit_transform(xtest[num_cat])

In [24]:
# model=LogisticRegression(max_iter=10000,solver='lbfgs')

In [25]:
# model.fit(xtrain,ytrain)

In [26]:
ytrain_pred=pipline.predict(xtrain)
ytest_pred=pipline.predict(xtest)

In [27]:
confusion_matrix(ytrain,ytrain_pred)

array([[26317,  1693],
       [ 2002,  5988]], dtype=int64)

In [28]:
confusion_matrix(ytest,ytest_pred)

array([[6560,  430],
       [ 519, 1491]], dtype=int64)

In [29]:
print("Precision Scorce of Train 0:",precision_score(ytrain,ytrain_pred,pos_label=0))
print("Precision Scorce of Train 1:",precision_score(ytrain,ytrain_pred,pos_label=1)) 

Precision Scorce of Train 0: 0.9293054133267418
Precision Scorce of Train 1: 0.7795859914073688


In [30]:
print("Precision Scorce of Test 0:",precision_score(ytest,ytest_pred,pos_label=0) )
print("Precision Scorce of Test 1:",precision_score(ytest,ytest_pred,pos_label=1) ) 

Precision Scorce of Test 0: 0.9266845599660969
Precision Scorce of Test 1: 0.7761582509109839


In [31]:
print("Recall Scorce of Train 0:",recall_score(ytrain,ytrain_pred,pos_label=0))
print("Recall Scorce of Train 1:",recall_score(ytrain,ytrain_pred,pos_label=1))

Recall Scorce of Train 0: 0.9395573009639414
Recall Scorce of Train 1: 0.7494367959949937


In [32]:
print("Recall Scorce of Test 0:",recall_score(ytest,ytest_pred,pos_label=0))
print("Recall Scorce of Test 1:",recall_score(ytest,ytest_pred,pos_label=1))

Recall Scorce of Test 0: 0.9384835479256081
Recall Scorce of Test 1: 0.7417910447761195


In [33]:
ytrain.value_counts()

loan_status
0    28010
1     7990
Name: count, dtype: int64

In [34]:
print("Accuary Scorce of Train:",accuracy_score(ytrain,ytrain_pred))
print("Accuary Scorce of Test:",accuracy_score(ytest,ytest_pred))

Accuary Scorce of Train: 0.8973611111111112
Accuary Scorce of Test: 0.8945555555555555


In [35]:
print("F1 Scorce of Train 0:",f1_score(ytrain,ytrain_pred,pos_label=0))
print("F1 Scorce of Train 1:",f1_score(ytrain,ytrain_pred,pos_label=1))

F1 Scorce of Train 0: 0.9344032381189086
F1 Scorce of Train 1: 0.7642141535320018


In [36]:
print("F1 Scorce of Test 0:",f1_score(ytest,ytest_pred,pos_label=0))
print("F1 Scorce of Test 1:",f1_score(ytest,ytest_pred,pos_label=1))

F1 Scorce of Test 0: 0.9325467339540835
F1 Scorce of Test 1: 0.7585856016280844


In [38]:
print(classification_report(ytrain,ytrain_pred))

              precision    recall  f1-score   support

           0       0.93      0.94      0.93     28010
           1       0.78      0.75      0.76      7990

    accuracy                           0.90     36000
   macro avg       0.85      0.84      0.85     36000
weighted avg       0.90      0.90      0.90     36000



In [39]:
print(classification_report(ytest,ytest_pred))

              precision    recall  f1-score   support

           0       0.93      0.94      0.93      6990
           1       0.78      0.74      0.76      2010

    accuracy                           0.89      9000
   macro avg       0.85      0.84      0.85      9000
weighted avg       0.89      0.89      0.89      9000

